In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import glob
import matplotlib.pyplot as plt
import plotly
import plotly.express as px
import plotly.graph_objs as go
import gzip
import h5py
import scanpy as sc
import scipy
import mira
import torch
import anndata as ad

# MIRA

In [ ]:
adata=sc.read_h5ad("/ix/djishnu/peasena/tf_perturbseq/20251008_bcl6ko_perturbseq/bcl6ko_cellranger_multi/outs/h5_files/merged.h5ad")

In [ ]:
adata.obs['batch'] = adata.obs['group'].apply(
    lambda x: 'donA' if 'donA' in x else ('donB' if 'donB' in x else x))

In [ ]:
model = mira.topics.make_model(
    adata.n_obs, adata.n_vars, # helps MIRA choose reasonable values for some hyperparameters which are not tuned.
    feature_type = 'expression',
    highly_variable_key='highly_variable',
    counts_layer='counts',
    categorical_covariates='batch'
)

In [ ]:
model.get_learning_rate_bounds(adata)

In [ ]:
model.set_learning_rates(1e-3, 0.25)
model.plot_learning_rate_bounds(figsize=(7,3))

# Hyperparameter Optimization

# Method 1: Gradient based

In [ ]:
#takes a long time (30min)
topic_contributions = mira.topics.gradient_tune(model, adata)

In [ ]:
with open('/ix/djishnu/peasena/tf_perturbseq/20251008_bcl6ko_perturbseq/bcl6ko_cellranger_multi/outs/mira/topic_contributions.txt', 'w') as file:
    file.write('\n'.join(str(topic) for topic in topic_contributions))

In [ ]:
NUM_TOPICS = 30

mira.pl.plot_topic_contributions(topic_contributions, NUM_TOPICS)

In [ ]:
#takes ~5-10min
model = model.set_params(num_topics = NUM_TOPICS).fit(adata)

In [ ]:
model.save('/ix/djishnu/peasena/tf_perturbseq/20251008_bcl6ko_perturbseq/bcl6ko_cellranger_multi/outs/mira/models/gradient_model_1.pth')

In [ ]:
#reload model if needed
model = mira.topic_model.load_model('/ix/djishnu/peasena/tf_perturbseq/20251008_bcl6ko_perturbseq/bcl6ko_cellranger_multi/outs/mira/models/gradient_model_1.pth')



# (Optional) Method 2: Bayesian Optimization

In [ ]:
tuner = mira.topics.BayesianTuner(
        model = model,
        n_jobs=2,
        save_name = '/ix/djishnu/peasena/tf_perturbseq/20251008_bcl6ko_perturbseq/bcl6ko_cellranger_multi/outs/mira/models/expression_model_tuner',
        #### IMPORTANT
        min_topics = 27, max_topics = 33, # tailor for your dataset!!!! (+/- 10 from the above topic number)
        #### See "Notes on min_topics, max_topics" above
        #storage = mira.topics.Redis() # if using REDIS backend for more (>5) processes
)

In [ ]:
#takes the longest (~2-4hrs)
tuner.fit(adata)

In [ ]:
ax = tuner.plot_intermediate_values(palette='Spectral_r',
                                   log_hue=True, figsize=(7,3))
# ax.set(ylim = (7e2, 7.7e2))

In [ ]:
tuner.plot_pareto_front(include_pruned_trials=False, label_pareto_front=True,
                       figsize = (5,5))

In [ ]:
model = tuner.fetch_best_weights()

In [ ]:
model.save('/ix/djishnu/peasena/tf_perturbseq/20251008_bcl6ko_perturbseq/bcl6ko_cellranger_multi/outs/mira/models/bayesian_model_1.pth')

In [ ]:
#uses topic model to predict for each cell
model.predict(adata)

In [ ]:
adata.write('/ix/djishnu/peasena/tf_perturbseq/20251008_bcl6ko_perturbseq/bcl6ko_cellranger_multi/outs/h5_files/merged_topic_modeling.h5ad')